# Feature importance for effect heterogeneity

**Purpose.** This notebook learns which features of the host economy before the opening modify the size of the effect of an opening on tourism receipts. The target is the effect in percentage points of GDP (`att_pp`) on the primary sample, with the economy as the grouping unit. The same analysis is repeated for the relative effect (`att_rel_pct`) and for the sensitivity samples. Each run reports the permutation importance averaged over the elastic net penalty path, the importance along the path, a group bootstrap band, the clusters of strongly correlated features with the importance of each cluster permuted as a whole, diagnostics and a usability verdict. The normalised importance of the primary run gives the feature weights of the transport cost in notebook 04.

**Input files.** `case_effects.csv`, `case_features_imputed.csv` and `imputation_report.csv` in the results directory, all written by notebook 02.

**Output files.** In the results directory: `feature_importance.csv`, `importance_diagnostics.json`, `transport_weights.csv` and `figure_inventory_03_feature_importance.csv`. In the figures directory: `importance_bars.png`, `importance_penalty_path.png`, `importance_selection_frequency.png` and `importance_rank_correlation.png`.

**How to run.** On the desktop that holds the real data, set `DTT_RUN_REAL=1` and execute all cells, for example with `jupyter nbconvert --to notebook --execute notebooks/03_feature_importance.ipynb`. To test the notebook on a simulated world, set `DTT_WORLD=sim` together with `DTT_SIM_DIR`, `DTT_RESULTS_DIR` and `DTT_FIGURE_DIR`, each pointing to a directory outside the repository, and run notebooks 00 and 02 first. A simulated run refuses an output directory inside the data, results or figures directory of the repository, never reads the real files and never writes to the results directory of the repository.

**Run time.** About 2 minutes in a simulated run and about 20 to 30 minutes on the real data, because the group bootstrap repeats the importance estimate for every resample of every run. A busy machine or a pause of the thermal guard makes the run longer.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "src" / "dtt" / "__init__.py").is_file())
sys.path.insert(0, str(_root / "src"))

from dtt import workflow

ctx = workflow.setup_run("03_feature_importance")
ctx.describe()

## Settings

The settings cell is the only place where the mode matters. In a simulated run (`FAST` is true) only the number of permutation repeats, the number of label permutations of the usability test, the number of random assignments of the economies to the folds and the number of bootstrap resamples are reduced. The stacked ensemble, the penalty grid, the folds, the clustering of the features and the usability criteria are the same in both modes.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.legend import Legend

from dtt import importance as imp
from dtt.cases import FEATURE_COLUMNS

FAST = ctx.mode == "sim"
N_REPEATS = 3 if FAST else 20
N_PERM = 20 if FAST else 100
N_BOOT = 5 if FAST else 100
CV_REPEATS = 2 if FAST else 3
BOOT_CV_REPEATS = 1
N_SPLITS = 5
WEIGHTING = "cv"
TEMPERATURE = 1.0
CLUSTER_THRESHOLD = 0.8
MIN_EPISODES = 8
MIN_ECONOMIES = 4
SHRINKAGE = 0.5
SEED = ctx.seeds["importance"]
BOOT_SEED = ctx.seeds["importance bootstrap"]
RUNS_TO_DO = [
    ("att_pp", "primary"),
    ("att_rel_pct", "primary"),
    ("att_pp", "parks_only"),
    ("att_pp", "first_episode"),
    ("att_pp", "all_feasible"),
]

MIN_N_PERM = 9
assert N_PERM >= MIN_N_PERM, "the label permutation test needs at least 9 permutations for a p-value of 0.10 to be reachable"

print(f"FAST: {FAST}")
print(f"Permutation repeats per feature and fold: {N_REPEATS}")
print(f"Label permutations of the usability test: {N_PERM} (at least {MIN_N_PERM} are needed when the diagnostics are computed)")
print(f"Bootstrap resamples: {N_BOOT}")
print(f"Random assignments of the economies to the outer folds: {CV_REPEATS} in each run, {BOOT_CV_REPEATS} in each bootstrap resample")
print(f"Outer grouped folds: {N_SPLITS}; penalty weighting: {WEIGHTING}, temperature {TEMPERATURE}")
print(f"Features with an absolute correlation above {CLUSTER_THRESHOLD} form a cluster that is also permuted as a whole")
print(f"Minimum size of a run: {MIN_EPISODES} episodes in at least {MIN_ECONOMIES} economies")
print(f"Shrinkage of the second weight vector towards equal weights: {SHRINKAGE}")
print(f"Seeds: importance {SEED}, importance bootstrap {BOOT_SEED}")

## Data

The features are the columns of the imputed feature table. A feature that notebook 02 dropped because too many of its values were missing is not used; the table below lists the dropped features with the reason from the imputation report. The rows are the estimated episodes. Each economy is one group, so no economy is split between the training part and the held-out part of a fold.

In [ ]:
ctx.require("case_effects.csv", "case_features_imputed.csv", "imputation_report.csv")
case_effects = ctx.read_table("case_effects.csv")
features_imputed = ctx.read_table("case_features_imputed.csv")
imputation_report = ctx.read_table("imputation_report.csv")

FEATURES = [c for c in features_imputed.columns if c in FEATURE_COLUMNS]
FEATURES_DROPPED = imputation_report.loc[imputation_report["dropped"].astype(bool), "feature"].tolist()
data = case_effects.drop(columns=[c for c in FEATURES if c in case_effects.columns]).merge(
    features_imputed[["case_id", *FEATURES]], on="case_id", how="inner", validate="one_to_one"
)

print(f"Episodes with an effect and features: {len(data)} in {data['iso3'].nunique()} economies")
print(f"Features used ({len(FEATURES)}): {', '.join(FEATURES)}")
if FEATURES_DROPPED:
    print("\nFeatures dropped for missing values:")
    print(imputation_report.loc[imputation_report["dropped"].astype(bool), ["feature", "n_missing", "share_missing", "reason"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
else:
    print("\nNo feature was dropped for missing values.")
imputed_counts = imputation_report.loc[
    ~imputation_report["dropped"].astype(bool) & (imputation_report["n_missing"] > 0), ["feature", "n_missing", "fill_value", "fill_source"]
]
if len(imputed_counts):
    print("\nFeatures with imputed values (fill_value is the mean of the values filled in; fill_source is the reference table, the other episodes or both):")
    print(imputed_counts.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

assert len(data) == len(case_effects) and data["case_id"].is_unique
assert np.isfinite(data[FEATURES].to_numpy(dtype=float)).all()
assert set(FEATURES) | set(FEATURES_DROPPED) == set(FEATURE_COLUMNS)
print("\nChecks passed: every effect has a complete feature vector.")

## Samples and runs

The primary sample consists of the episodes with a good pre-opening fit; episodes whose pre-opening path is reproduced exactly while the donors outnumber the pre-opening years are not part of it. The sensitivity samples are the primary sample restricted to theme parks and multi-park resorts, the first episode of each economy within the primary sample, and all feasible episodes. The sample of all feasible episodes also holds the episodes that were set aside for an exact fit that is not unique; for these episodes `att_pp` is one point of the identified range between `att_pp_min` and `att_pp_max`. A sensitivity sample that contains exactly the episodes of an earlier run with the same outcome reuses that run. A run with too few episodes or too few economies is skipped and the reason is printed.

In [ ]:
primary = data["sample_primary"].to_numpy(dtype=bool)
SAMPLES = {
    "primary": primary,
    "parks_only": primary & data["parks_only"].to_numpy(dtype=bool),
    "first_episode": primary & data["first_episode"].to_numpy(dtype=bool),
    "all_feasible": np.ones(len(data), dtype=bool),
}
rows = []
for outcome, sample in RUNS_TO_DO:
    use = SAMPLES[sample] & np.isfinite(data[outcome].to_numpy(dtype=float))
    rows.append(
        {
            "outcome": outcome,
            "sample": sample,
            "episodes": int(use.sum()),
            "economies": int(data.loc[use, "iso3"].nunique()),
        }
    )
run_table = pd.DataFrame(rows)
print(run_table.to_string(index=False))
n_primary = int(primary.sum())
print(
    f"\nThe primary sample has {n_primary} episodes in {data.loc[primary, 'iso3'].nunique()} economies; "
    f"all feasible episodes number {len(data)} in {data['iso3'].nunique()} economies."
)
non_unique_exact = data["fit_exact"].to_numpy(dtype=bool) & (data["n_pre"].to_numpy() < data["n_donors"].to_numpy())
print(f"Episodes with an exact fit that is not unique, which are in the sample of all feasible episodes only: {int(non_unique_exact.sum())}.")
n_rel_missing = int(data["att_rel_pct"].isna().sum())
print(f"The relative effect att_rel_pct is missing for {n_rel_missing} of {len(data)} episodes; such episodes are left out of the runs on it.")
assert not (primary & non_unique_exact).any()
assert (data["att_rel_pct"].isna() | np.isfinite(data["att_rel_pct"])).all()
for outcome, sample, n, g in run_table[["outcome", "sample", "episodes", "economies"]].itertuples(index=False):
    if n < MIN_EPISODES or g < MIN_ECONOMIES:
        print(f"Run {outcome} | {sample} will be skipped: {n} episodes in {g} economies.")

## Usability criteria

The usability rule is part of `dtt.importance`. The cell below prints the criteria from the module constants.

In [ ]:
print(
    "The importance vector of a run is marked as usable when the grouped cross-validated R2 of the stack is at least "
    f"{imp.MIN_CV_R2:.2f}, the label permutation p-value is at most {imp.MAX_PERM_P_VALUE:.2f}, "
    f"the median Pearson correlation of the normalised importance vectors of two random halves of the economies is at least {imp.MIN_SPLIT_HALF_CORRELATION:.2f} "
    f"and at least {imp.MIN_CASES} cases are present."
)

The label permutation p-value and the split-half correlation are computed only in the main importance estimate of each run. The bootstrap resamples and the single-penalty runs of the penalty path skip them, so a verdict for those estimates would always be not usable and is not used. A verdict of not usable does not stop the analysis. The importance weights are still saved to `transport_weights.csv` and reported, and notebook 04 still computes the importance-weighted transport. The route rule that was fixed in advance (`decide_route` in `dtt.impact`, applied in notebook 05) requires a usable verdict among its conditions, so with a verdict of not usable the importance-weighted route cannot be the primary route.

## Running the importance analysis

The helper below runs the penalty-averaged importance and the group bootstrap for one outcome and one sample. The importance is the increase of the held-out squared error when a feature is permuted, averaged over repeated permutations, over the outer grouped folds and over the penalties, with penalty weights that decrease with the cross-validated error. It is reported in the units of the squared outcome and, after setting negative values to zero and dividing by the sum, as a share (`permutation_normalised`). The coefficient path is the mean absolute standardised elastic net coefficient, and the selection frequency is the share of fold and penalty fits in which a coefficient is not zero. The economies are assigned at random to the outer folds, and the assignment is repeated several times with different seeds; the repeats count equally. The bootstrap resamples economies with replacement and reports the median and the 10th and 90th percentiles of the normalised importance.

Features whose absolute correlation exceeds the threshold of the settings cell form a cluster. The columns of a cluster are also permuted together, and the increase of the held-out error after that joint permutation is reported as `cluster_importance` for every member of the cluster. A feature that is not in a cluster has a cluster of its own, and its cluster importance equals its permutation importance. The cluster labels are integers that count the clusters in the order in which they first appear among the features of the run.

In [ ]:
RUNS: dict[tuple[str, str], dict] = {}
_SEEN: dict[tuple, tuple[str, str]] = {}


def duration_text(seconds: float) -> str:
    """Format a duration for the printed timings.

    Parameters
    ----------
    seconds : float
        Duration in seconds.

    Returns
    -------
    str
        Whole seconds below one and a half minutes, otherwise minutes with one decimal.
    """
    return f"{seconds:.0f} seconds" if seconds < 90.0 else f"{seconds / 60.0:.1f} minutes"


def run_importance(outcome: str, sample: str) -> dict:
    """Run the penalty-averaged importance and its group bootstrap for one outcome and one sample.

    Parameters
    ----------
    outcome : str
        Column of the effects table used as target, ``att_pp`` or ``att_rel_pct``.
    sample : str
        Key of ``SAMPLES``.

    Returns
    -------
    dict
        Keys ``outcome``, ``sample``, ``n_cases``, ``n_economies``, ``skipped``, ``reason`` and ``same_as``; for a run
        that was not skipped also ``result`` (an ``ImportanceResult``), ``band`` (bootstrap table or None), ``table``
        (one row per feature with the importance measures, the clusters and the bootstrap band), ``X``, ``y`` and
        ``groups``. The key ``same_as`` names the earlier run whose episodes and outcome are identical, in which case
        its results are reused; otherwise the keys ``seconds_main`` and ``seconds_bootstrap`` hold the computation
        times of the importance estimate and of the bootstrap.
    """
    mask = SAMPLES[sample] & np.isfinite(data[outcome].to_numpy(dtype=float))
    frame = data.loc[mask]
    record = {
        "outcome": outcome,
        "sample": sample,
        "n_cases": len(frame),
        "n_economies": int(frame["iso3"].nunique()),
        "skipped": False,
        "reason": "",
        "same_as": None,
    }
    if record["n_cases"] < MIN_EPISODES or record["n_economies"] < MIN_ECONOMIES:
        record["skipped"] = True
        record["reason"] = (
            f"{record['n_cases']} episodes in {record['n_economies']} economies, but a run needs at least "
            f"{MIN_EPISODES} episodes in at least {MIN_ECONOMIES} economies"
        )
        return record
    if not float(np.var(frame[outcome].to_numpy(dtype=float))) > 0.0:
        record["skipped"] = True
        record["reason"] = "the outcome is constant in this sample"
        return record
    key = (outcome, tuple(sorted(frame["case_id"])))
    if key in _SEEN:
        earlier = RUNS[_SEEN[key]]
        record.update({k: earlier[k] for k in ("result", "band", "table", "X", "y", "groups")})
        record["same_as"] = _SEEN[key]
        return record
    X, y, groups = frame[FEATURES], frame[outcome], frame["iso3"]
    print(f"Computing {outcome} | {sample}: {record['n_cases']} episodes, {N_BOOT} bootstrap resamples", flush=True)
    if ctx.guard:
        ctx.guard()
    started = time.perf_counter()
    result = imp.lambda_averaged_importance(
        X, y, groups, n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED, weighting=WEIGHTING,
        temperature=TEMPERATURE, n_perm=N_PERM, guard=ctx.guard, cv_repeats=CV_REPEATS,
        cluster_threshold=CLUSTER_THRESHOLD, compute_diagnostics=True,
    )
    record["seconds_main"] = time.perf_counter() - started
    started = time.perf_counter()
    try:
        band = imp.bootstrap_importance(
            X, y, groups, n_boot=N_BOOT, random_state=BOOT_SEED, guard=ctx.guard, n_splits=N_SPLITS,
            n_repeats=N_REPEATS, weighting=WEIGHTING, temperature=TEMPERATURE, cv_repeats=BOOT_CV_REPEATS,
        )
    except ValueError as error:
        band = None
        record["reason"] = f"bootstrap not available: {error}"
    record["seconds_bootstrap"] = time.perf_counter() - started
    table = result.to_frame()
    if band is not None:
        table = table.join(band.rename(columns={"median": "boot_median", "p10": "boot_p10", "p90": "boot_p90"}))
    else:
        for name in ("boot_median", "boot_p10", "boot_p90"):
            table[name] = np.nan
    table = table.rename_axis("feature").reset_index()
    record.update({"result": result, "band": band, "table": table, "X": X, "y": y, "groups": groups})
    _SEEN[key] = (outcome, sample)
    return record


def cluster_members(table: pd.DataFrame) -> list[list[str]]:
    """List the clusters that hold more than one feature.

    Parameters
    ----------
    table : pandas.DataFrame
        Importance table of a run with the columns ``feature`` and ``cluster``.

    Returns
    -------
    list of list of str
        The features of each cluster with at least two members, in the order of the cluster labels.
    """
    members = table.groupby("cluster", sort=True)["feature"].apply(list)
    return [list(names) for names in members if len(names) > 1]


def report_run(record: dict) -> None:
    """Print the sample size, the importance table, the clusters, the diagnostics and the verdict of a run.

    Parameters
    ----------
    record : dict
        Output of :func:`run_importance`.
    """
    name = f"{record['outcome']} | {record['sample']}"
    print(f"Run {name}: {record['n_cases']} episodes in {record['n_economies']} economies")
    if record["skipped"]:
        print(f"  Skipped: {record['reason']}.")
        return
    if record["same_as"] is not None:
        print(f"  The episodes and the outcome are identical to the run {record['same_as'][0]} | {record['same_as'][1]}; its results are reused.")
        print(f"  Verdict: {'usable' if record['result'].diagnostics['usable'] else 'not usable'}.")
        return
    if record["reason"]:
        print(f"  Note: {record['reason']}.")
    d = record["result"].diagnostics
    shown = record["table"].sort_values("permutation_normalised", ascending=False)
    print(shown.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    groups_found = cluster_members(record["table"])
    if groups_found:
        print(f"  Clusters of features with an absolute correlation above {CLUSTER_THRESHOLD}: " + "; ".join(", ".join(names) for names in groups_found))
    else:
        print(f"  No two features have an absolute correlation above {CLUSTER_THRESHOLD}, so every feature is a cluster of its own.")
    print("  Diagnostics:")
    print(f"    grouped cross-validated R2 of the stack: {d['cv_r2_stack']:.3f} (best single learner {d['cv_r2_best_single']:.3f}, intercept only {d['cv_r2_null']:.3f})")
    print(f"    label permutation p-value: {d['perm_p_value']:.3f}")
    print(f"    median split-half Pearson correlation of the normalised importance: {d['split_half_correlation']:.3f}")
    print(f"    cases: {d['n_cases']}, features: {d['n_features']}")
    print(
        f"  Computation time: {duration_text(record['seconds_main'])} for the importance estimate "
        f"and {duration_text(record['seconds_bootstrap'])} for the bootstrap"
    )
    print(f"  Verdict: {'usable' if d['usable'] else 'not usable'}.")
    for reason in d["reasons"]:
        print(f"    Reason: {reason}.")

## Primary run

The primary run uses the effect in percentage points of GDP on the primary sample. The importance table is sorted by the normalised permutation importance. The diagnostics and the verdict follow the table.

In [ ]:
started_cell = time.perf_counter()
RUNS[("att_pp", "primary")] = run_importance("att_pp", "primary")
primary_run = RUNS[("att_pp", "primary")]
report_run(primary_run)
print(f"\nElapsed time of this cell: {duration_text(time.perf_counter() - started_cell)}")

## Secondary and sensitivity runs

The same procedure is repeated for the relative effect on the primary sample and for the sensitivity samples with the effect in percentage points of GDP as the outcome.

In [ ]:
started_cell = time.perf_counter()
for outcome, sample in RUNS_TO_DO[1:]:
    RUNS[(outcome, sample)] = run_importance(outcome, sample)
    report_run(RUNS[(outcome, sample)])
    print()
done = [key for key, r in RUNS.items() if not r["skipped"]]
print(f"{len(done)} of {len(RUNS)} runs completed.")
for key, r in RUNS.items():
    if r["skipped"]:
        print(f"Skipped: {key[0]} | {key[1]} ({r['reason']}).")
print(f"\nElapsed time of this cell: {duration_text(time.perf_counter() - started_cell)}")

## Transport weights

The transport weights are computed from the primary run. The first weight vector is the normalised permutation importance, the second mixes it with equal weights, and the third is equal weights. All three sum to one. The weights in `transport_weights.csv` come from all primary episodes and serve the transport to the target economy; notebook 04 relearns the weights inside every validation fold. The normalised permutation importance is that of the single features, so clustered features are treated like all others: each member of a cluster keeps its own share, and the cluster importance is reported next to it but does not enter the weights. If the primary run was skipped, no importance exists, and all three vectors are equal weights.

In [ ]:
d_features = len(FEATURES)
if primary_run["skipped"]:
    w_importance = np.full(d_features, 1.0 / d_features)
    w_shrunk = np.full(d_features, 1.0 / d_features)
    print("The primary run was skipped, so the importance weights are equal weights.")
else:
    w_importance = primary_run["result"].weights(0.0)
    w_shrunk = primary_run["result"].weights(SHRINKAGE)
transport_weights = pd.DataFrame(
    {
        "feature": FEATURES,
        "weight_importance": w_importance,
        "weight_shrunk": w_shrunk,
        "weight_uniform": np.full(d_features, 1.0 / d_features),
    }
)
ctx.save_table(transport_weights, "transport_weights.csv")
print(transport_weights.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

assert list(transport_weights.columns) == ["feature", "weight_importance", "weight_shrunk", "weight_uniform"]
for column in ("weight_importance", "weight_shrunk", "weight_uniform"):
    assert np.isclose(transport_weights[column].sum(), 1.0) and (transport_weights[column] >= 0).all()
print("\nChecks passed: each weight vector is non-negative and sums to one.")

## Saving the importance tables and diagnostics

The importance table has one row per run and feature. The diagnostics file is keyed by outcome and sample and holds the diagnostics of each run together with the number of cases and features, the features used and the features dropped. A skipped run has an entry with the verdict not usable and the reason for skipping it.

In [ ]:
frames = []
diagnostics_out = {}
for (outcome, sample), record in RUNS.items():
    key = f"{outcome}|{sample}"
    base = {
        "n_cases": record["n_cases"],
        "n_economies": record["n_economies"],
        "n_features": len(FEATURES),
        "features_used": FEATURES,
        "features_dropped": FEATURES_DROPPED,
        "mode": ctx.mode,
    }
    if record["skipped"]:
        diagnostics_out[key] = {**base, "skipped": True, "usable": False, "reasons": [f"run skipped: {record['reason']}"]}
        continue
    table = record["table"].set_index("feature").reindex(FEATURES).reset_index()
    frames.append(table.assign(outcome=outcome, sample=sample))
    diagnostics_out[key] = {
        **record["result"].diagnostics,
        **base,
        "skipped": False,
        "same_as_run": None if record["same_as"] is None else "|".join(record["same_as"]),
        "n_repeats": N_REPEATS,
        "n_perm": N_PERM,
        "n_boot": N_BOOT,
        "cv_repeats": CV_REPEATS,
        "bootstrap_cv_repeats": BOOT_CV_REPEATS,
        "cluster_threshold": CLUSTER_THRESHOLD,
        "clusters": cluster_members(record["table"]),
    }
columns = [
    "outcome", "sample", "feature", "permutation", "permutation_normalised", "coefficient_path", "selection_frequency",
    "boot_median", "boot_p10", "boot_p90", "cluster", "cluster_importance",
]
feature_importance = pd.concat(frames, ignore_index=True)[columns] if frames else pd.DataFrame(columns=columns)
ctx.save_table(feature_importance, "feature_importance.csv")
ctx.save_json(diagnostics_out, "importance_diagnostics.json")

for key, entry in diagnostics_out.items():
    verdict = "usable" if entry["usable"] else "not usable"
    print(f"{key}: {entry['n_cases']} episodes, {verdict}" + ("" if not entry["skipped"] else " (skipped)"))

assert list(feature_importance.columns) == columns
assert len(feature_importance) == len(frames) * len(FEATURES)
if len(feature_importance):
    assert feature_importance.groupby(["outcome", "sample"])["permutation_normalised"].sum().sub(1.0).abs().max() < 1e-9
    assert feature_importance["selection_frequency"].between(0.0, 1.0).all()
    with_band = feature_importance.dropna(subset=["boot_median"])
    assert ((with_band["boot_p10"] <= with_band["boot_median"] + 1e-12) & (with_band["boot_median"] <= with_band["boot_p90"] + 1e-12)).all()
    assert np.isfinite(feature_importance["cluster_importance"]).all()
    for _, group in feature_importance.groupby(["outcome", "sample"]):
        size = group.groupby("cluster")["feature"].transform("size")
        assert np.allclose(group.loc[size == 1, "cluster_importance"], group.loc[size == 1, "permutation"])
        assert (group.groupby("cluster")["cluster_importance"].nunique() == 1).all()
assert all(isinstance(entry["usable"], bool) for entry in diagnostics_out.values())
for entry in diagnostics_out.values():
    if not entry["skipped"]:
        assert "split_half_correlation" in entry and "split_half_spearman" not in entry and "clusters" in entry
print("\nChecks passed: normalised importances sum to one in every run, the bootstrap bands bracket their medians, and the cluster columns are consistent.")

## Figure: importance of the primary run

The bars show the normalised permutation importance of the primary run, sorted with the largest at the top. Hatched bars and the cluster letter after a feature name mark features that belong to a cluster of correlated features. The diamonds are the medians of the bootstrap resamples and the lines span their 10th to 90th percentiles. The importance measures predictive relevance of a feature for the estimated effects in this sample; it does not give the direction of the relationship.

In [ ]:
def legend_below(ax: plt.Axes, **kwargs: object) -> Legend:
    """Place the legend of an axes under its x axis label, whatever the size of the figure.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes with a finished x axis label and labelled artists.
    **kwargs
        Passed to ``Axes.legend``.

    Returns
    -------
    matplotlib.legend.Legend
        The legend.
    """
    fig = ax.figure
    fig.canvas.draw()
    label_box = ax.xaxis.label.get_window_extent(fig.canvas.get_renderer())
    y_label = ax.transAxes.inverted().transform((0.0, label_box.y0))[1]
    return ax.legend(loc="upper center", bbox_to_anchor=(0.5, y_label - 0.03), **kwargs)


if primary_run["skipped"]:
    print("Figure not drawn: the primary run was skipped.")
else:
    table = primary_run["table"].sort_values("permutation_normalised", ascending=True).reset_index(drop=True)
    ORDER = table["feature"].tolist()[::-1]
    y_pos = np.arange(len(table))
    cluster_groups = cluster_members(table)
    letter_of = {name: chr(ord("A") + k) for k, names in enumerate(cluster_groups) for name in names}
    clustered = table["feature"].isin(list(letter_of)).to_numpy()
    fig, ax = plt.subplots(figsize=(8.4, 0.55 * len(table) + 2.4))
    if (~clustered).any():
        ax.barh(
            y_pos[~clustered], table.loc[~clustered, "permutation_normalised"], color="0.78", edgecolor="0.0", height=0.62,
            label="Normalised permutation importance",
        )
    if clustered.any():
        ax.barh(
            y_pos[clustered], table.loc[clustered, "permutation_normalised"], color="0.78", edgecolor="0.0", height=0.62,
            hatch="///", label="Normalised permutation importance, feature in a cluster",
        )
    if primary_run["band"] is not None:
        lower = np.clip(table["boot_median"] - table["boot_p10"], 0.0, None)
        upper = np.clip(table["boot_p90"] - table["boot_median"], 0.0, None)
        ax.errorbar(
            table["boot_median"], y_pos, xerr=[lower, upper], fmt="D", color="0.0", markerfacecolor="white", markersize=5,
            capsize=3, linewidth=1.3, label="Bootstrap median with 10th to 90th percentile band",
        )
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{name} (cluster {letter_of[name]})" if name in letter_of else name for name in table["feature"]])
    ax.set_xlim(left=0.0)
    ax.set_xlabel("Normalised permutation importance (share of the total)")
    ax.set_ylabel("Feature")
    usable = primary_run["result"].diagnostics["usable"]
    ax.set_title(
        f"Importance for the effect in percentage points, primary sample ({primary_run['n_cases']} episodes)\n"
        f"Usability verdict: {'usable' if usable else 'not usable'}"
    )
    legend_below(ax, fontsize=8)
    ctx.save_figure(fig, "importance_bars.png")
    top = table.sort_values("permutation_normalised", ascending=False).head(3)
    print("Largest normalised importance: " + ", ".join(f"{r.feature} {r.permutation_normalised:.3f}" for r in top.itertuples()))
    by_feature = table.set_index("feature")
    for k, names in enumerate(cluster_groups):
        rows = by_feature.loc[names]
        print(
            f"Cluster {chr(ord('A') + k)}: {', '.join(names)}. Permutation importance of the single features: "
            + ", ".join(f"{value:.4f}" for value in rows["permutation"])
            + f"; of the cluster permuted as a whole: {float(rows['cluster_importance'].iloc[0]):.4f} (units of the squared outcome)."
        )
    if not cluster_groups:
        print("No cluster of correlated features: every feature is shown on its own.")

## Importance across the penalty path

The penalty-averaged importance is a weighted mean of the importance at each penalty of the grid. The penalties refer to standardised data, and the grid starts at the smallest penalty for which every elastic net coefficient is zero. To show the importance at each penalty, the analysis is repeated with a grid that holds a single penalty, with all other settings and seeds unchanged. The label permutation test, the split-half check and the clustering are not repeated for these runs. The weighted mean of these single-penalty importances with the penalty weights reproduces the averaged importance of the primary run, which the cell checks.

In [ ]:
def penalty_path(record: dict) -> pd.DataFrame:
    """Permutation importance at every penalty of the grid of a finished run.

    Parameters
    ----------
    record : dict
        Output of :func:`run_importance` for a run that was not skipped.

    Returns
    -------
    pandas.DataFrame
        One row per penalty (index ``penalty``, in the order of the grid of the run) and one column per feature, with the
        increase of the held-out mean squared error after permuting the feature.
    """
    result = record["result"]
    rows = []
    for penalty in result.lambdas:
        if ctx.guard:
            ctx.guard()
        single = imp.lambda_averaged_importance(
            record["X"], record["y"], record["groups"], lambdas=[float(penalty)], n_splits=N_SPLITS, n_repeats=N_REPEATS,
            random_state=SEED, weighting=WEIGHTING, temperature=TEMPERATURE, guard=ctx.guard, cv_repeats=CV_REPEATS,
            cluster_threshold=None, compute_diagnostics=False,
        )
        rows.append(single.permutation.to_numpy(dtype=float))
    return pd.DataFrame(rows, index=pd.Index(result.lambdas, name="penalty"), columns=result.feature_names)


PATH = None
if primary_run["skipped"]:
    print(f"The penalty path is not computed because the primary run was skipped ({primary_run['reason']}).")
else:
    started_path = time.perf_counter()
    PATH = penalty_path(primary_run)
    print(f"Penalty path computed in {duration_text(time.perf_counter() - started_path)}")
    result = primary_run["result"]
    averaged = result.lambda_weights @ PATH.to_numpy()
    worst = float(np.max(np.abs(averaged - result.permutation.to_numpy())))
    scale = float(np.max(np.abs(result.permutation.to_numpy())))
    print(f"Penalty grid: {len(PATH)} penalties, log10 from {np.log10(PATH.index.max()):.2f} down to {np.log10(PATH.index.min()):.2f}")
    path_summary = pd.DataFrame(
        {
            "log10 penalty": np.log10(result.lambdas),
            "cv mean squared error": result.cv_mse_by_lambda,
            "penalty weight": result.lambda_weights,
        }
    )
    print(path_summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(
        f"\nLargest difference between the weighted mean of the single-penalty importances and the averaged importance: "
        f"{worst:.2e} ({100 * worst / scale if scale > 0 else 0.0:.3f} percent of the largest importance)."
    )
    assert worst <= 0.01 * scale if scale > 0 else True
    print("Checks passed: the single-penalty importances reproduce the averaged importance.")

## Figure: importance along the penalty path

The upper panel shows, for each feature, the permutation importance at each penalty of the grid, against the base-10 logarithm of the standardised penalty. Small penalties are on the left and large penalties on the right, where the model approaches the intercept-only model and every importance is close to zero. The lower panel gives the weight of each penalty in the average. A penalty with a small weight has little influence on the averaged importance, however large the importance at that penalty is.

In [ ]:
if PATH is None:
    print("Figure not drawn: the primary run was skipped.")
else:
    log_penalty = np.log10(PATH.index.to_numpy(dtype=float))
    order = np.argsort(log_penalty)
    weights_by_penalty = primary_run["result"].lambda_weights
    colours, line_styles, markers = ["0.0", "0.35", "0.6"], ["-", "--", ":"], ["o", "s", "^", "v", "D", "P", "X", "*", "<", ">"]
    fig, (ax, ax_w) = plt.subplots(
        2, 1, figsize=(9.6, 7.6), sharex=True, gridspec_kw={"height_ratios": [3.0, 1.2], "hspace": 0.08}
    )
    for i, feature in enumerate(PATH.columns):
        ax.plot(
            log_penalty[order], PATH[feature].to_numpy()[order], color=colours[i % 3], linestyle=line_styles[(i // 3) % 3],
            marker=markers[i % len(markers)], markersize=4.5, linewidth=1.3, label=feature,
        )
    ax.axhline(0.0, color="0.5", linewidth=0.8)
    ax.set_ylabel("Permutation importance (increase of held-out\nmean squared error, squared percentage points)")
    ax.set_title("Permutation importance of each feature along the penalty path (primary run)")
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
    spacing = float(np.min(np.diff(np.sort(log_penalty)))) if len(log_penalty) > 1 else 0.5
    ax_w.bar(log_penalty[order], weights_by_penalty[order], width=0.8 * spacing, color="0.55", edgecolor="0.15")
    ax_w.set_ylabel("Penalty weight (share)")
    ax_w.set_xlabel("Base-10 logarithm of the standardised penalty")
    ctx.save_figure(fig, "importance_penalty_path.png")
    peak = PATH.abs().max(axis=1).idxmax()
    print(f"Largest single-penalty importance at log10 penalty {np.log10(peak):.2f}; penalty weights sum to {weights_by_penalty.sum():.3f}.")

## Figure: selection frequency

The bars give the share of fold and penalty fits in which the elastic net coefficient of the feature is not zero. The features are in the order of the importance figure.

In [ ]:
if primary_run["skipped"]:
    print("Figure not drawn: the primary run was skipped.")
else:
    freq = primary_run["table"].set_index("feature").loc[ORDER[::-1], "selection_frequency"]
    y_pos = np.arange(len(freq))
    fig, ax = plt.subplots(figsize=(8.0, 0.5 * len(freq) + 2.0))
    ax.barh(y_pos, freq.to_numpy(), color="0.7", edgecolor="0.0", height=0.62)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(freq.index)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel("Share of fold and penalty fits with a non-zero elastic net coefficient")
    ax.set_ylabel("Feature")
    ax.set_title("Selection frequency of each feature (primary run)")
    ctx.save_figure(fig, "importance_selection_frequency.png")
    print("Selection frequency: " + ", ".join(f"{name} {value:.2f}" for name, value in freq.items()))

## Rank comparison across runs

The table gives the rank of each feature by normalised permutation importance in every completed run, with rank 1 for the most important feature and average ranks for ties. A run that reuses the results of an earlier run with the same episodes and outcome is not shown again. The figure shows the Spearman rank correlation between the importance vectors of the runs. A high correlation means that the runs order the features alike.

In [ ]:
completed = {key: r for key, r in RUNS.items() if not r["skipped"] and r["same_as"] is None}
labels = {key: f"{key[0]}, {key[1]}" for key in completed}
for key, r in RUNS.items():
    if not r["skipped"] and r["same_as"] is not None:
        print(f"Run {key[0]} | {key[1]} has the same episodes and outcome as {r['same_as'][0]} | {r['same_as'][1]} and is not shown again.")
if len(completed) == 0:
    print("No run was completed, so no ranks are shown.")
else:
    importance_wide = pd.DataFrame(
        {labels[key]: r["table"].set_index("feature").loc[FEATURES, "permutation_normalised"] for key, r in completed.items()}
    )
    ranks = importance_wide.rank(ascending=False, method="average")
    print("Rank of each feature by normalised permutation importance (1 is the most important):")
    print(ranks.sort_values(ranks.columns[0]).to_string(float_format=lambda v: f"{v:.1f}"))
    if len(completed) < 2:
        print("\nFewer than two runs were completed, so no correlation matrix is drawn.")
    else:
        correlation = importance_wide.corr(method="spearman")
        fig, ax = plt.subplots(figsize=(1.35 * len(correlation) + 3.4, 1.2 * len(correlation) + 2.6))
        light_grays = LinearSegmentedColormap.from_list("light_grays", ["1.0", "0.3"])
        shown = ax.imshow(correlation.to_numpy(dtype=float), cmap=light_grays, vmin=-1.0, vmax=1.0)
        ax.grid(False)
        ax.set_xticks(np.arange(len(correlation)))
        ax.set_yticks(np.arange(len(correlation)))
        ax.set_xticklabels(correlation.columns, rotation=30, ha="right")
        ax.set_yticklabels(correlation.index)
        for i in range(len(correlation)):
            for j in range(len(correlation)):
                value = correlation.iat[i, j]
                text = "n/a" if not np.isfinite(value) else f"{value:.2f}"
                ax.text(j, i, text, ha="center", va="center", fontsize=9, color="white" if np.isfinite(value) and value > 0.5 else "black")
        ax.set_xlabel("Outcome and sample of the run")
        ax.set_ylabel("Outcome and sample of the run")
        ax.set_title("Spearman correlation of the importance vectors across runs")
        colour_bar = fig.colorbar(shown, ax=ax, fraction=0.046, pad=0.04)
        colour_bar.set_label("Spearman rank correlation")
        ctx.save_figure(fig, "importance_rank_correlation.png")
        off_diagonal = correlation.to_numpy(dtype=float)[~np.eye(len(correlation), dtype=bool)]
        print(f"\nSpearman correlations between runs: minimum {np.nanmin(off_diagonal):.2f}, median {np.nanmedian(off_diagonal):.2f}, maximum {np.nanmax(off_diagonal):.2f}.")

## Scope and limits

Permutation importance measures how much a feature helps to predict the estimated effects of the episodes in this sample. It does not give the direction of a relationship and it is not a causal statement about what would happen if the feature were changed. When features are correlated, the permutation importance of a single feature can be small because the others carry the same information. The cluster importance, which permutes the correlated features together, shows their joint contribution. The transport weights use the single-feature importance and not the cluster importance, so the weights of a cluster of correlated features can differ from what its joint importance would give. The sample is small, so the importance vector is noisy, and the bootstrap band shows only part of the uncertainty, because it resamples economies but not the estimation error of the effects themselves. Each effect is an estimate with its own placebo standard error, and that error is not carried into the importance analysis.

The usability verdict combines the criteria printed above: whether the model predicts held-out economies better than chance, whether the label permutation test rejects an unrelated target, whether the normalised importance vectors of two random halves of the economies agree, and whether enough episodes are present. A verdict of not usable means that at least one criterion failed, and the reasons printed with each run say which. The split-half correlation is a Pearson correlation of the two importance vectors, so it is high when both halves single out the same features and low when the largest importance moves from one feature to another between the halves. Each half holds half of the economies and is analysed with the light configuration of `dtt.importance`, which uses fewer penalties, smaller trees, fewer permutation repeats and one random assignment of the economies to the folds. The weights are still written to `transport_weights.csv` and reported, and the route rule fixed in advance decides which route is primary in notebook 05. The sensitivity runs show whether the ordering of the features depends on the outcome scale, on the kind of attraction, on the number of episodes per economy or on the pre-opening fit criterion, because the sample of all feasible episodes also holds the episodes with an exact fit that is not unique; a low correlation between runs is a reason to read the importance ordering with caution.